In [15]:
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
)

from trl import SFTTrainer, SFTConfig

In [22]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

QUICK_TEST = False

OUTPUT_DIR = "outputs/mr_en_lora"


In [23]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [24]:
# ============================================================
# 3. 4-bit configuration
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [26]:
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

In [27]:
dataset = load_dataset(
    "json",
    data_files={
        "train": "processed/mr_en/train.jsonl",
        "validation": "processed/mr_en/val.jsonl",
    }
)

In [28]:
def format_example(example):

    messages = example["prompt"] + example["completion"]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }

In [29]:
dataset = dataset.map(
    format_example
)


In [30]:
if QUICK_TEST:

    dataset["train"] = dataset["train"].select(
        range(min(20, len(dataset["train"])))
    )

    dataset["validation"] = dataset["validation"].select(
        range(min(5, len(dataset["validation"])))
    )

    print("\nQUICK TEST MODE")
    print("Training examples:", len(dataset["train"]))
    print("Validation examples:", len(dataset["validation"]))

In [31]:
# ============================================================
# 8. LoRA configuration
# ============================================================

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

In [32]:
# ============================================================
# 9. Training arguments
# ============================================================

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    num_train_epochs=1,

    logging_steps=1,

    save_strategy="steps",
    save_steps=20,

    eval_strategy="steps",
    eval_steps=20,

    fp16=True,

    gradient_checkpointing=True,

    report_to="none",

    optim="paged_adamw_8bit",

    remove_unused_columns=False,
)


In [33]:
# ============================================================
# 10. Trainer
# ============================================================

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    num_train_epochs=1,

    logging_steps=1,

    save_strategy="steps",
    save_steps=20,

    eval_strategy="steps",
    eval_steps=20,

    fp16=False,
    bf16=False,
    gradient_checkpointing=True,

    report_to="none",

    optim="paged_adamw_8bit",

    remove_unused_columns=False,

    # These belong here in the newer TRL API
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,

    args=sft_config,

    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],

    processing_class=tokenizer,

    peft_config=lora_config,
)


Tokenizing train dataset:   0%|          | 0/8811 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8811 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8811 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8811 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1535 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1535 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1535 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1535 [00:00<?, ? examples/s]

In [34]:
trainer.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 12. Save adapter
# ============================================================

print("\nSaving LoRA adapter...")

trainer.save_model(
    f"{OUTPUT_DIR}/final"
)

tokenizer.save_pretrained(
    f"{OUTPUT_DIR}/final"
)

print("\nTraining completed!")